In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import duckdb


In [2]:

# Load the Parquet file using DuckDB
con = duckdb.connect()
df = con.execute("""
    SELECT "Time Stamp", Load
    FROM read_parquet('./../../1_LIB/nyiso/nyiso_parquet/**/*.parquet')
""").df()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [3]:
#need to double check zone coords
zone_coords = {
    "CAPITL": (42.65, -73.75),
    "CENTRL": (43.05, -76.15),
    "DUNWOD": (41.01, -73.78),
    "GENESE": (43.17, -77.61),
    "HUD VL": (41.70, -73.93),
    "MHK VL": (42.10, -75.91),
    "MILLWD": (41.13, -73.78),
    "N.Y.C.": (40.71, -74.01),
    "NORTH": (44.70, -73.45),
    "WEST": (42.89, -78.87)
}

In [4]:
df

,Time Stamp,Load
0,2001-05-26 00:00:00,985
1,2001-05-26 00:00:00,1461
2,2001-05-26 00:00:00,443
3,2001-05-26 00:00:00,830
4,2001-05-26 00:00:00,970
...,...,...
29303437,2025-10-01 00:30:00,644
29303438,2025-10-01 00:30:00,205
29303439,2025-10-01 00:30:00,5080
29303440,2025-10-01 00:30:00,527


In [5]:
df_total_load = df.groupby("Time Stamp", as_index=False)["Load"].sum()
df_total_load.rename(columns={"Time Stamp": "Time"}, inplace=True)


In [6]:
df_total_load

,Time,Load
0,2001-05-26 00:00:00,13859
1,2001-05-26 00:00:50,13761
2,2001-05-26 00:05:20,13718
3,2001-05-26 00:06:50,13622
4,2001-05-26 00:11:50,13551
...,...,...
2710904,2025-10-01 00:10:00,14720
2710905,2025-10-01 00:15:00,14663
2710906,2025-10-01 00:20:00,14628
2710907,2025-10-01 00:25:00,14612


In [9]:
cols = [
    'datetime',
    'temp_2m [degF]',
    'relative_humidity [percent]',
    'precip_1hr [inch]',
    'avg_wind_speed_merge [mile/hr]',
    'solar_insolation [W/m^2]'
]

col_str = ", ".join(f'"{c}"' for c in cols)

query = f"""
    SELECT {col_str}
    FROM read_parquet('./../../mesonet_master/*.parquet')
"""

res = con.execute(query)
reader = res.fetch_record_batch(50000)   # stream in chunks
first_batch = next(iter(reader))

mesonet_chunk = first_batch.to_pandas()
print(mesonet_chunk.columns)
print(mesonet_chunk)


Index(['datetime', 'temp_2m [degF]', 'relative_humidity [percent]',
       'precip_1hr [inch]', 'avg_wind_speed_merge [mile/hr]',
       'solar_insolation [W/m^2]'],
      dtype='object')
                       datetime  temp_2m [degF]  relative_humidity [percent]  \
0     2015-08-10 00:00:00-04:00             NaN                         95.2   
1     2015-08-10 00:05:00-04:00             NaN                         95.6   
2     2015-08-10 00:10:00-04:00             NaN                         96.6   
3     2015-08-10 00:15:00-04:00             NaN                         96.7   
4     2015-08-10 00:20:00-04:00             NaN                         96.5   
...                         ...             ...                          ...   
49995 2016-01-31 15:35:00-05:00            51.0                         43.3   
49996 2016-01-31 15:40:00-05:00            50.9                         43.5   
49997 2016-01-31 15:45:00-05:00            50.8                         43.3   
49998 2016-0

In [10]:
import duckdb

con = duckdb.connect()

result = con.execute("""
    SELECT COUNT(*) 
    FROM read_parquet('./../../mesonet_master/*.parquet')
""").fetchone()

print("Total rows:", result[0])


Total rows: 1057304


In [11]:
import duckdb
import pandas as pd

# Columns you want for load prediction
load_features = [
    "station",
    "datetime",
    "temp_2m [degF]",
    "apparent_temperature [degF]",
    "relative_humidity [percent]",
    "precip_1hr [inch]",
    "avg_wind_speed_merge [mile/hr]",
    "solar_insolation [W/m^2]"
]

# Build SQL query string
col_str = ", ".join(f'"{c}"' for c in load_features)

query = f"""
    SELECT
        DATE_TRUNC('day', CAST(datetime AS TIMESTAMP)) AS day,
        AVG("temp_2m [degF]") AS avg_temp,
        AVG("apparent_temperature [degF]") AS avg_apparent_temp,
        AVG("relative_humidity [percent]") AS avg_humidity,
        SUM("precip_1hr [inch]") AS total_precip,
        AVG("avg_wind_speed_merge [mile/hr]") AS avg_wind_speed,
        AVG("solar_insolation [W/m^2]") AS avg_solar
    FROM read_parquet('./../../mesonet_master/*.parquet')
    GROUP BY DATE_TRUNC('day', CAST(datetime AS TIMESTAMP))
    ORDER BY day
"""


# Execute query and convert to Pandas
daily_df = con.execute(query).df()
print(daily_df.head())


         day   avg_temp  avg_apparent_temp  avg_humidity  total_precip  \
0 2015-08-10        NaN                NaN     79.095111         0.000   
1 2015-08-11  70.291848          70.291848     84.264621         9.627   
2 2015-08-12  68.592254          68.592254     84.210915         2.089   
3 2015-08-13  65.112500          65.112500     86.718056         1.264   
4 2015-08-14  68.948727          69.314182     83.041818         0.000   

   avg_wind_speed   avg_solar  
0        2.580088  264.623894  
1        2.220578   75.158845  
2        1.314789  206.007042  
3        1.332639  198.003472  
4        2.464000  236.625455  


In [12]:
print(daily_df.describe())

                              day     avg_temp  avg_apparent_temp  \
count                        3673  3662.000000        3662.000000   
mean   2020-09-12 03:58:45.510482    47.752128          45.252035   
min           2015-08-10 00:00:00    -7.461806         -18.178819   
25%           2018-03-10 00:00:00    33.621615          28.979948   
50%           2020-09-13 00:00:00    48.953299          47.756944   
75%           2023-03-27 00:00:00    63.312066          63.300174   
max           2025-09-30 00:00:00    81.500000          83.286111   
std                           NaN    17.558509          20.315465   

       avg_humidity  total_precip  avg_wind_speed    avg_solar  
count   3633.000000   3316.000000     3663.000000  3663.000000  
mean      74.629086      1.266046        5.887280   144.465345  
min       28.179514      0.000000        0.270000     0.000000  
25%       66.874306      0.000000        3.883681    66.822273  
50%       75.752778      0.036000        5.353125   1

In [13]:
import pandas as pd

res = con.execute("SELECT * FROM read_parquet('./../../mesonet_master/*.parquet')")
reader = res.fetch_record_batch(5)
chunk = next(iter(reader))
mesonet_chunk = chunk.to_pandas()
print(mesonet_chunk.columns)


Index(['datetime', 'latitude [degrees_north]', 'longitude [degrees_east]',
       'elevation [feet]', 'temp_2m [degF]', 'temp_9m [degF]',
       'apparent_temperature [degF]', 'relative_humidity [percent]',
       'dewpoint [degF]', 'precip_incremental [inch]', 'precip_local [inch]',
       'precip_max_intensity [inch/hour]', 'precip_1hr [inch]',
       'avg_wind_speed_prop [mile/hr]', 'max_wind_speed_prop [mile/hr]',
       'wind_speed_stddev_prop [mile/hr]', 'wind_direction_prop [degrees]',
       'wind_direction_stddev_prop [degrees]',
       'avg_wind_speed_sonic [mile/hr]', 'max_wind_speed_sonic [mile/hr]',
       'wind_speed_stddev_sonic [mile/hr]', 'wind_direction_sonic [degrees]',
       'wind_direction_stddev_sonic [degrees]',
       'avg_wind_speed_merge [mile/hr]', 'max_wind_speed_merge [mile/hr]',
       'wind_speed_stddev_merge [mile/hr]', 'wind_direction_merge [degrees]',
       'wind_direction_stddev_merge [degrees]', 'solar_insolation [W/m^2]',
       'station_pressure 

In [14]:
# Ensure the 'Time' column is a datetime type and set it as the index
df_total_load['Time'] = pd.to_datetime(df_total_load['Time'])
df_total_load.set_index('Time', inplace=True)

# Average hourly load
df_hourly = df_total_load.resample('D').mean().reset_index()
df_hourly = df_hourly.dropna().reset_index()



In [15]:
weather_df = daily_df.rename(columns={'day': 'Time'})

merged = pd.merge(df_hourly, weather_df, on='Time', how='inner')


In [16]:
merged

,index,Time,Load,avg_temp,avg_apparent_temp,avg_humidity,total_precip,avg_wind_speed,avg_solar
0,5189,2015-08-10,20993.663194,NaN,NaN,79.095111,0.000,2.580088,264.623894
1,5190,2015-08-11,21176.321918,70.291848,70.291848,84.264621,9.627,2.220578,75.158845
2,5191,2015-08-12,21014.796552,68.592254,68.592254,84.210915,2.089,1.314789,206.007042
3,5192,2015-08-13,20496.397924,65.112500,65.112500,86.718056,1.264,1.332639,198.003472
4,5193,2015-08-14,21091.59375,68.948727,69.314182,83.041818,0.000,2.464000,236.625455
...,...,...,...,...,...,...,...,...,...
3668,8889,2025-09-26,17650.0,62.413889,62.413889,82.935417,0.000,3.000347,164.180556
3669,8890,2025-09-27,15957.758389,61.732639,61.732639,81.177083,0.000,3.108681,133.166667
3670,8891,2025-09-28,16220.506897,65.307292,65.307639,74.670833,0.000,4.107292,189.861111
3671,8892,2025-09-29,16641.556314,56.697569,56.690625,73.394444,0.000,1.621875,159.427083


In [17]:
df_hourly

,index,Time,Load
0,0,2001-05-26,14717.505435
1,1,2001-05-27,14126.594667
2,2,2001-05-28,14443.625974
3,3,2001-05-29,16909.186528
4,4,2001-05-30,16543.040541
...,...,...,...
8859,8890,2025-09-27,15957.758389
8860,8891,2025-09-28,16220.506897
8861,8892,2025-09-29,16641.556314
8862,8893,2025-09-30,16199.019737


In [18]:
df_total_load = merged

In [19]:
# Split the data based on the year
train_data = df_total_load[df_total_load['Time'].dt.year.between(2001, 2021)]
val_data = df_total_load[df_total_load['Time'].dt.year == 2022]
test_data = df_total_load[df_total_load['Time'].dt.year.isin([2023, 2024, 2025])]

# Print the sizes of each split
print(f"Training data size: {len(train_data)}")
print(f"Validation data size: {len(val_data)}")
print(f"Testing data size: {len(test_data)}")

Training data size: 2304
Validation data size: 365
Testing data size: 1004


In [20]:
train_data

,index,Time,Load,avg_temp,avg_apparent_temp,avg_humidity,total_precip,avg_wind_speed,avg_solar
0,5189,2015-08-10,20993.663194,NaN,NaN,79.095111,0.000,2.580088,264.623894
1,5190,2015-08-11,21176.321918,70.291848,70.291848,84.264621,9.627,2.220578,75.158845
2,5191,2015-08-12,21014.796552,68.592254,68.592254,84.210915,2.089,1.314789,206.007042
3,5192,2015-08-13,20496.397924,65.112500,65.112500,86.718056,1.264,1.332639,198.003472
4,5193,2015-08-14,21091.59375,68.948727,69.314182,83.041818,0.000,2.464000,236.625455
...,...,...,...,...,...,...,...,...,...
2299,7520,2021-12-27,17683.236111,29.223611,22.860417,86.773611,2.134,7.501736,13.972222
2300,7521,2021-12-28,17160.955479,34.878472,32.512500,79.289931,2.747,3.615278,47.100694
2301,7522,2021-12-29,17152.659722,34.742014,33.597917,96.814583,0.402,2.446875,24.638889
2302,7523,2021-12-30,16728.510417,37.996181,36.680556,98.525347,0.200,2.753125,21.465278


In [21]:
train_data = train_data.dropna()
val_data = val_data.dropna()
test_data = test_data.dropna()
scaler = StandardScaler()
y_scaler = StandardScaler()
train_scaled = scaler.fit_transform(pd.DataFrame(train_data.drop(['Time', 'index'], axis=1)))
val_scaled = scaler.transform(pd.DataFrame(val_data.drop(['Time', 'index'], axis=1)))
test_scaled = scaler.transform(pd.DataFrame(test_data.drop(['Time', 'index'], axis=1)))

y_train_scaled = y_scaler.fit_transform(pd.DataFrame(train_data['Load']))
y_val_scaled = y_scaler.transform(pd.DataFrame(val_data['Load']))
y_test_scaled = y_scaler.transform(pd.DataFrame(test_data['Load']))

In [22]:
def create_dataset(X, y, time_steps=1):
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        v = X[i:i + time_steps]
        Xs.append(v)
        ys.append(y[i + time_steps])
    return np.array(Xs), np.array(ys)

TIME_STEPS = 5
X_train, y_train = create_dataset(train_scaled, y_train_scaled, TIME_STEPS)
X_val, y_val = create_dataset(val_scaled, y_val_scaled, TIME_STEPS)
X_test, y_test = create_dataset(test_scaled, y_test_scaled, TIME_STEPS)
print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

(1929, 5, 7) (1929, 1)
(360, 5, 7) (360, 1)
(987, 5, 7) (987, 1)


In [23]:
print("NaNs in X_train:", np.isnan(X_train).sum())
print("NaNs in X_val:", np.isnan(X_val).sum())
print("NaNs in y_train:", np.isnan(y_train).sum())
print("NaNs in y_val:", np.isnan(y_val).sum())

NaNs in X_train: 0
NaNs in X_val: 0
NaNs in y_train: 0
NaNs in y_val: 0


In [ ]:
subset_size = 40000
val_subset = 8749
X_train_sub = X_train[:subset_size]
y_train_sub = y_train[:subset_size]
X_val_sub = X_val[:val_subset]
y_val_sub = y_val[:val_subset]


In [ ]:
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error
# best_mae = float('inf')
# best_model = None

# for C in [0.1, 1, 10]:
#     for gamma in ['scale', 0.01, 0.001]:
#         for epsilon in [0.01, 0.1, 0.5, 1.0]:
#             print(f"C={C}, gamma={gamma}, epsilon={epsilon}")
#             model = SVR(kernel='rbf', C=C, gamma=gamma, epsilon=epsilon)
#             model.fit(X_train_sub.reshape(X_train_sub.shape[0], -1), y_train_sub)
#             preds = model.predict(X_val_sub.reshape(X_val_sub.shape[0], -1))
#             mae = mean_absolute_error(y_val_sub, preds)
#             print(f"val MAE={mae:.3f}")

#             if mae < best_mae:
#                 best_mae = mae
#                 best_model = model

# print("Best params found:", best_model.get_params())


Best params found: {'C': 10, 'cache_size': 200, 'coef0': 0.0, 'degree': 3, 'epsilon': 0.01, 'gamma': 0.01, 'kernel': 'rbf', 'max_iter': -1, 'shrinking': True, 'tol': 0.001, 'verbose': False}

Best mae: 0.04994650252799576

In [ ]:
best_params = {'C': 10, 'cache_size': 200, 'coef0': 0.0, 'degree': 3, 'epsilon': 0.01, 'gamma': 0.01, 'kernel': 'rbf', 'max_iter': -1, 'shrinking': True, 'tol': 0.001, 'verbose': False}
best_model = SVR(**best_params)
best_model.fit(X_train.reshape(X_train.shape[0], -1), y_train)

In [ ]:

# Make predictions
train_pred = best_model.predict(X_train.reshape(X_train.shape[0], -1))
train_pred = y_scaler.inverse_transform(train_pred.reshape(-1, 1))
y_train_inv = y_scaler.inverse_transform(y_train.reshape(-1, 1))

val_pred = best_model.predict(X_val.reshape(X_val.shape[0], -1))
val_pred = y_scaler.inverse_transform(val_pred.reshape(-1, 1))
y_val_inv = y_scaler.inverse_transform(y_val.reshape(-1, 1))

test_pred = best_model.predict(X_test.reshape(X_test.shape[0], -1))
test_pred = y_scaler.inverse_transform(test_pred.reshape(-1, 1))
y_test_inv = y_scaler.inverse_transform(y_test.reshape(-1, 1))


In [ ]:
# Evaluate the model
mae_train = mean_absolute_error(y_train_inv, train_pred)
mae_test = mean_absolute_error(y_test_inv, test_pred)
print("Mean Absolute Error on Training Data:", mae_train)
print("Mean Absolute Error on Testing Data:", mae_test)

In [ ]:

from sklearn.metrics import mean_absolute_percentage_error


mape_train = mean_absolute_percentage_error(y_train_inv, train_pred)
mape_test = mean_absolute_percentage_error(y_test_inv, test_pred)
print("Mean Absolute Error on Training Data:", mape_train)
print("Mean Absolute Error on Testing Data:", mape_test)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import numpy as np
import matplotlib as mpl
mpl.rcParams['animation.embed_limit'] = 100 

y_true = y_test_inv
y_pred = test_pred
x = np.arange(len(y_true))

window = 100  # how many points to show at once
lag = 1       # number of steps prediction lags behind

fig, ax = plt.subplots(figsize=(10,6))
line_true, = ax.plot([], [], label="True Values", color="blue", alpha=0.7)
line_pred, = ax.plot([], [], label="Predictions", color="red", alpha=0.7)
ax.set_ylim(min(y_true.min(), y_pred.min())*0.95, max(y_true.max(), y_pred.max())*1.05)
ax.set_xlabel("Index")
ax.set_ylabel("Load")
ax.set_title("Predictions vs True Values (Trailing Window)")
ax.legend()

def update(frame):
    start = max(0, frame - window)
    end = frame
    line_true.set_data(x[start:end], y_true[start:end])
    
    # Prediction lags behind true values
    pred_start = max(0, frame - window - lag)
    pred_end = max(0, frame - lag)
    line_pred.set_data(x[pred_start:pred_end], y_pred[pred_start:pred_end])
    
    ax.set_xlim(x[start], x[end-1] if end > start else x[start]+1)
    return line_true, line_pred

ani = FuncAnimation(
    fig, update,
    frames=range(0, len(x), 10),   # every 10th frame
    interval=20, blit=True
)
HTML(ani.to_jshtml())



In [ ]:
mape = np.mean(np.abs((y_test_inv - test_pred) / y_test_inv)) * 100
print(f"Testing MAPE: {mape:.2f}%")

eps = 1e-6
mape = np.mean(np.abs((y_train_inv - train_pred) / (y_train_inv + eps))) * 100
print(f"Training MAPE: {mape:.2f}%")

mape = np.mean(np.abs((y_val_inv - val_pred) / y_val_inv)) * 100
print(f"Val MAPE: {mape:.2f}%")



In [ ]:
from sklearn.metrics import r2_score

rmse = np.sqrt(mean_squared_error(y_test_inv, test_pred))
print(f"Testing RMSE: {rmse}")

r2 = r2_score(y_test_inv, test_pred)
print(f"Testing R2: {r2}")

In [ ]:
import joblib

joblib.dump(best_model, "svr_daily_model.joblib")


In [ ]:
subset_start = 0
subset_end = 1000
plt.figure(figsize=(12,6))
plt.plot(y_test_inv[subset_start: subset_end], label="True Load", color="black", alpha=0.7)
plt.plot(test_pred[subset_start: subset_end], label="Predictions", color="orange", alpha=0.7)
plt.xlabel("Time Index")
plt.ylabel("Load (MW)")
plt.title("SVR Predictions vs True Load")
plt.legend()
plt.show()

In [ ]:
# from datetime import datetime
# import matplotlib.pyplot as plt
# import meteostat
# from meteostat import Point, Daily, Hourly

# start = datetime(2018, 1, 1, 0, 0)
# end = datetime(2018, 1, 1, 12, 0)

# # Create Point for Vancouver, BC
# vancouver = Point(42.65, -73.75)

# # Get daily data for 2018
# data = Hourly(vancouver, start, end)
# data = data.fetch()

# # Plot line chart including average, minimum and maximum temperature
# data